# Lesson 1: making Markov chains

This lesson can be downloaded as a notebook, a notebook for colab and a python file [here](https://marmote.gitlabpages.inria.fr/marmote/python_downloads.html)

This C++ notebook follows the same pedagogical order as the Python lesson while using the official Marmote C++ API.

**Import the modules**

In [1]:
#ifdef _WIN32
#pragma cling add_include_path("C:/Users/assia/miniconda3/envs/jcpp-win/Library/include")
#pragma cling add_include_path("C:/Users/assia/miniconda3/envs/jcpp-win/Library/include/marmoteCore")
#pragma cling add_include_path("C:/Users/assia/miniconda3/envs/jcpp-win/Library/include/marmoteMarkovChain")
#pragma cling add_library_path("C:/Users/assia/miniconda3/envs/jcpp-win/Library/bin")
#pragma cling load("marmoteCore.dll")
#pragma cling load("marmoteMarkovChain.dll")
#else
#pragma cling add_include_path("/home/assia/miniconda3/envs/xeus-cpp-env/include")
#pragma cling add_include_path("/home/assia/miniconda3/envs/xeus-cpp-env/include/marmoteCore")
#pragma cling add_include_path("/home/assia/miniconda3/envs/xeus-cpp-env/include/marmoteMarkovChain")
#pragma cling add_library_path("/home/assia/miniconda3/envs/xeus-cpp-env/lib")
#pragma cling load("libmarmoteCore.so")
#pragma cling load("libmarmoteMarkovChain.so")
#endif

The Python lesson imports `marmote.core` and `marmote.markovchain`. In C++, the corresponding Marmote headers are made explicit below.

In [2]:
// --- Standard C++ utilities used in this notebook ---
#include <iostream>
#include <iomanip>
#include <string>
#include <vector>

// --- Marmote headers used in this lesson ---
#include <marmoteCore/marmoteCore>
#include <marmoteMarkovChain/marmoteMarkovChain>

// --- Convenience declarations for the cells below ---
using namespace std;
using namespace marmote;

A Markov Chain is composed of
+ a **state space**
+ a **transition structure** (probability matrix or infinitesimal generator)
+ an **initial distribution** of the state

In this first lesson, we show various ways of creating and inspecting Markov chains in C++.

## First example: a discrete-time Markov chain with 3 states

### State space

In [3]:
// The Python notebook first creates the vector of state labels.
double states[3] = {0.0, 1.0, 2.0};
stateType n = 3;

### Transition structure

We now create a `FullMatrix`, declare it as a discrete-time transition structure, and fill its entries with the same probabilities as in Python.

In [4]:
// Create a full transition matrix and declare that it represents a discrete-time chain.
FullMatrix* P = new FullMatrix(n);
P->set_type(DISCRETE);

// Fill the transition probabilities.
P->setEntry(0,0,0.25);
P->setEntry(0,1,0.50);
P->setEntry(0,2,0.25);
P->setEntry(1,0,0.40);
P->setEntry(1,1,0.20);
P->setEntry(1,2,0.40);
P->setEntry(2,0,0.40);
P->setEntry(2,1,0.30);
P->setEntry(2,2,0.30);

// Inspect the matrix.
P->Write(&cout, "", FORMAT_MARMOTE);

         0          0 2.500000e-01
         0          1 5.000000e-01
         0          2 2.500000e-01
         1          0 4.000000e-01
         1          1 2.000000e-01
         1          2 4.000000e-01
         2          0 4.000000e-01
         2          1 3.000000e-01
         2          2 3.000000e-01


### Initial distribution

As in Python, we build a `DiscreteDistribution` on the same three states.

In [5]:
// Create the initial distribution.
double initial_prob[3] = {0.2, 0.2, 0.6};
DiscreteDistribution* initial = new DiscreteDistribution(3, states, initial_prob);
cout << *initial << endl;

DiscreteDistribution (Object at 0x60071c8104d0)Discrete distribution values { 0 1 2 } probas {      0.2      0.2      0.6 }


### The Markov chain

The `MarkovChain` object is then built directly from the transition structure.

In [6]:
// Build the Markov chain from the transition matrix.
MarkovChain* c1 = new MarkovChain(P);
c1->set_init_distribution(initial);
c1->set_model_name("Demo_Discrete");

// Print the chain and inspect its time type.
c1->Write(&cout);
cout << "Type = " << c1->type() << " ; DISCRETE = " << DISCRETE << endl;

discrete sparse
3
         0          0 2.500000e-01
         0          1 5.000000e-01
         0          2 2.500000e-01
         1          0 4.000000e-01
         1          1 2.000000e-01
         1          2 4.000000e-01
         2          0 4.000000e-01
         2          1 3.000000e-01
         2          2 3.000000e-01
stop
discrete values { 0 1 2 } probas {      0.2      0.2      0.6 } 
Type = 0 ; DISCRETE = 0


### Input/Output of Markov Chains and transition structures

As in Python, the transition structure can be exported in several textual formats.

In [7]:
// Display the same matrix in several exchange formats.
cout << c1->generator()->toString(FORMAT_MATLAB_SPARSE) << endl;
cout << c1->generator()->toString(FORMAT_NUMPY) << endl;
cout << c1->generator()->toString(FORMAT_R) << endl;
cout << c1->generator()->toString(FORMAT_MAPLE) << endl;

theRows = [ 0 0 0 1 1 1 2 2 2]';
theColumns = [ 0 1 2 0 1 2 0 1 2]';
theValues = [         0.25          0.5         0.25          0.4          0.2          0.4          0.4          0.3          0.3]';

mc_matrix=np.array([
[0.25, 0.5, 0.25],
[0.4, 0.2, 0.4],
[0.4, 0.3, 0.3]
], dtype=float)

mc_matrix=matrix(c(0.25, 0.5, 0.25, 0.4, 0.2, 0.4, 0.4, 0.3, 0.3), nrow=3, byrow=TRUE)

_matrix := Matrix( 3, 3, {
(1, 1)=2.500000e-01,
(1, 2)=5.000000e-01,
(1, 3)=2.500000e-01,
(2, 1)=4.000000e-01,
(2, 2)=2.000000e-01,
(2, 3)=4.000000e-01,
(3, 1)=4.000000e-01,
(3, 2)=3.000000e-01,
(3, 3)=3.000000e-01
}, storage=rectangular):



## Second example with a continuous-time Markov chain

This time we use a `SparseMatrix` as transition structure support.

In [8]:
// Create a sparse infinitesimal generator with 6 states.
SparseMatrix* Q = new SparseMatrix(6);
Q->set_type(CONTINUOUS);
Q->setEntry(0,1,1.0);
Q->setEntry(0,0,-1.0);
for (stateType i = 1; i < 6; i++) {
    if (i > 0) {
        Q->setEntry(i,0,1.0);
        Q->addToEntry(i,i,-1.0);
    }
    if (i < 5) {
        Q->setEntry(i,i+1,1.0);
        Q->addToEntry(i,i,-1.0);
    }
}
Q->Write(&cout, "", FORMAT_MARMOTE);

// Build the continuous-time chain.
MarkovChain* c2 = new MarkovChain(Q);
c2->set_init_distribution(initial);
c2->set_model_name("Demo_Continuous");
c2->Write(&cout);
cout << "Type = " << c2->type() << " ; CONTINUOUS = " << CONTINUOUS << endl;

         0          1 1.000000e+00
         0          0 -1.000000e+00
         1          0 1.000000e+00
         1          1 -2.000000e+00
         1          2 1.000000e+00
         2          0 1.000000e+00
         2          2 -2.000000e+00
         2          3 1.000000e+00
         3          0 1.000000e+00
         3          3 -2.000000e+00
         3          4 1.000000e+00
         4          0 1.000000e+00
         4          4 -2.000000e+00
         4          5 1.000000e+00
         5          0 1.000000e+00
         5          5 -1.000000e+00
continuous sparse
6
         0          1 1.000000e+00
         0          0 -1.000000e+00
         1          0 1.000000e+00
         1          1 -2.000000e+00
         1          2 1.000000e+00
         2          0 1.000000e+00
         2          2 -2.000000e+00
         2          3 1.000000e+00
         3          0 1.000000e+00
         3          3 -2.000000e+00
         3          4 1.000000e+00
         4          0 1.0

## Transformation of Markov chains

The Python notebook presents the two standard transformations of continuous-time chains into discrete-time ones: **uniformization** and **embedding**.

In [9]:
// Uniformize the chain with the automatically chosen rate.
MarkovChain* c2uni = c2->Uniformize();
c2uni->Write(&cout);
cout << "Uniformization rate = " << c2uni->generator()->uniformization_rate() << endl;

// Redo uniformization with a larger rate.
MarkovChain* c2uni2 = new MarkovChain(c2->generator()->Uniformize(4.0));
cout << c2uni2->generator()->toString(FORMAT_NUMPY) << endl;

// Embed the chain at jump times.
MarkovChain* c2embed = c2->Embed();
cout << c2embed->generator()->toString(FORMAT_MARMOTE) << endl;

discrete sparse
6
         0          1 5.000000e-01
         0          0 5.000000e-01
         1          0 5.000000e-01
         1          2 5.000000e-01
         2          0 5.000000e-01
         2          3 5.000000e-01
         3          0 5.000000e-01
         3          4 5.000000e-01
         4          0 5.000000e-01
         4          5 5.000000e-01
         5          0 5.000000e-01
         5          5 5.000000e-01
stop
0
Uniformization rate = 2.000000e+00
mc_matrix=np.array([
[0.75, 0.25, 0, 0, 0, 0],
[0.25, 0.5, 0.25, 0, 0, 0],
[0.25, 0, 0.5, 0.25, 0, 0],
[0.25, 0, 0, 0.5, 0.25, 0],
[0.25, 0, 0, 0, 0.5, 0.25],
[0.25, 0, 0, 0, 0, 0.75]
], dtype=float)

         0          1 1.000000e+00
         1          0 5.000000e-01
         1          2 5.000000e-01
         2          0 5.000000e-01
         2          3 5.000000e-01
         3          0 5.000000e-01
         3          4 5.000000e-01
         4          0 5.000000e-01
         4          5 5.000000e-01
    

In [10]:
// Release the Markov chain objects created in this lesson.
delete c2embed;
delete c2uni2;
delete c2uni;
delete c2;
delete c1;